# Observing the Sun (2/3): The Sun in EUV
This notebook is one of three notebooks used to explore solar observations across the visible end ultraviolet spectrum. This notebook focuses on the extreme ultraviolet (EUV) imagery. The other two notebooks view imagary of sunspots and magnetic active regions, and the white light imagery of the solar corona. These other notebooks should be in the same directory.

Another notebook you will want to reference looks at the GOES X-Ray flux from the total disc od the Sun.

The goal of all these notebooks together is to compare the imagery of the Sun and see how the features seen in each type of view correspond to the features in the other views.

If you are working in a group, each member of the group may want to run a different and you can compare one to the other if the dates are aligned.

Learning Goals
- View solar features, particularly Sunspots and active regions across different types of imagary.
- Explore solar imagery at different phases in the solar cycle.

We have been continously observing the Sun using Extreme UltraViolet (EUV) imagery for over 3 Decades and with multiple spacecraft missions. One of the current missions is the Solar Dynamic Observatory [(SDO)](https://sdo.gsfc.nasa.gov) which observes the Sun in many different specral regions - extreme-UV and visible - and using different techniques. For this notebook, you will look at the EUV data collected using the [AIA](https://aia.lmsal.com) instrument on SDO.

|Wavelength|Source|Temp|Region
|------------|------------|--------|--------|
| 94 Angstroms| Fe XVIII| 6 million K |Flaring Region|
| 131 Angstroms| Fe VIII, XX, XXIII| 10 million K| Flaring Region|
| 171 Angstroms| Fe IX| 60,000 K|quiet corona|
| 193 Angstroms| Fe XII, XXIV| 1 million K|corona and flare plasma|
| 211 Angstroms| Fe XIV| 2 million K|active regions|
| 304 Angstroms| He II| 50,000 K|Chromosphere and transition region|
| 335 Angstroms| Fe XVI| 2.5 million K|active region corona|
| 1600 Angstroms| C IV & continum|10,000 K| upper photosphere|
| 1700 Angstroms| continum| 4500 K| photosphere|

* From [SDO Wikipedia entry](https://en.wikipedia.org/wiki/Solar_Dynamics_Observatory)


## Activity 0: Getting Started
The next few cells load some required packages including the ["AstroPy"](https://www.astropy.org) and ["SunPy"](https://sunpy.org) which include routines specifically for astronomy and solar physics.

In [ ]:
# Standard imports
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime, timedelta
#import warnings
import sys
import gc
import os

In [ ]:
### Do we need to load this packages in the HelioCloud environment?
%pip install -q sunpy aiapy solarmach
# Astropy imports
import astropy.units as u
from astropy.time import Time
from astropy.visualization import time_support, quantity_support
from astropy.visualization import ImageNormalize, AsinhStretch, PowerStretch
from astropy.table import Table

# SunPy imports
import sunpy
import sunpy.map
import sunpy.timeseries
from sunpy.net import Fido, attrs as a
from sunpy.time import TimeRange
from sunpy.coordinates import frames

# Additional packages
from aiapy.calibrate import correct_degradation
from solarmach import SolarMACH

%pip install -q sunpy_cloudcatalog
import sunpy_cloudcatalog

from sunpy.net import Fido, attrs as a
from sunpy_cloudcatalog import Bucket, DataID

We also need to setup a directory path in the environment.

In [ ]:
import pathlib
DATA_DIR = pathlib.Path('./data')
DATA_DIR.mkdir(exist_ok=True)

## Activity 1: Loading and Displaying an AIA image
We will use a two functions for the SunPy package that get solar imagery from a variety of spacecraft and missions.  These functions are
- [Fido.search](https://docs.sunpy.org/en/stable/tutorial/acquiring_data/index.html#searching-for-data) - searches for data files based on date range, instrument, and wavelength and returns the file details of the search.
- [Fido.fetch](https://docs.sunpy.org/en/stable/tutorial/acquiring_data/index.html#downloading-data) - actually downloads the data found by the search to the working directory for display and manipulation.

The next cell searches for data around a specific date and downloads the files it finds. The data it searches has the following parameters
- Time: close to start of the day on 2024-05-08
- wavelength 171 Angstroms from the AIA instrument - not the key word in the Fido.search command "a.Wavelength".

In [ ]:
time_start = Time('2024-05-08 12:00:00')

q_aia171 = Fido.search(
    a.Time(time_start - 30*u.minute, time_start + 30*u.minute, near=time_start),
    a.Instrument.aia,
    a.Wavelength(171*u.AA)
)

file_aia171 = Fido.fetch(q_aia171, path=DATA_DIR / 'AIA' / '{file}')

We can then use the [sunpy.map.Map function](https://docs.sunpy.org/en/stable/generated/api/sunpy.map.Map.html#sunpy.map.Map) to generate an image that can be displayed using the matplotlib function "imshow".

In [ ]:
map171 = sunpy.map.Map(file_aia171)
plt.imshow(map171.data, cmap = 'Blues_r',)

### Discussion
- How would you describe this image?  Is the contrast good?

We can add a color map normalization to the above line. See how that works in the cell below. We can use a non-linear scaling, the ["Asinh Stretch"](https://docs.astropy.org/en/stable/api/astropy.visualization.AsinhStretch.html) function, to bring out important features.

In [ ]:
plt.imshow(map171.data, cmap='Blues_r',norm=ImageNormalize(vmin=1,
                                             vmax=np.percentile(map171.data, 99.5),
                                             stretch=AsinhStretch()))

## Explore other wavelengths
You can use the previous few cells to choose a different wavelengths availabe from AIA. We suggest using a chromosphere and flare region related wavelengths. You can also assign different [matplotlib color maps](https://matplotlib.org/stable/users/explain/colors/colormaps.html) for 'cmap' keyword for different wavelengths.  

In [ ]:
time_start = Time('2024-05-8 12:00:00')

# Wavelength 1: 94 A
q_aia_1 = Fido.search(
    a.Time(time_start - 30*u.minute, time_start + 30*u.minute, near=time_start),
    a.Instrument.aia,
    a.Wavelength(94*u.AA)
)

file_aia_1 = Fido.fetch(q_aia_1, path=DATA_DIR / 'AIA' / '{file}')
map_1 = sunpy.map.Map(file_aia_1)


# Wavelength 2: 171 A
q_aia_2 = Fido.search(
    a.Time(time_start - 30*u.minute, time_start + 30*u.minute, near=time_start),
    a.Instrument.aia,
    a.Wavelength(171*u.AA)
)

file_aia_2 = Fido.fetch(q_aia_2, path=DATA_DIR / 'AIA' / '{file}')
map_2 = sunpy.map.Map(file_aia_2)

# Wavelength 3: 1700 A
q_aia_3 = Fido.search(
    a.Time(time_start - 30*u.minute, time_start + 30*u.minute, near=time_start),
    a.Instrument.aia,
    a.Wavelength(1700*u.AA)
)
file_aia_3 = Fido.fetch(q_aia_3, path=DATA_DIR / 'AIA' / '{file}')
map_3 = sunpy.map.Map(file_aia_3)

# Display all three images
fig, ax = plt.subplots(1,3, figsize = (20,60))
ax[0].imshow(map_1.data, cmap='Blues_r',norm=ImageNormalize(vmin=1,
                                             vmax=np.percentile(map_1.data, 99.5),
                                             stretch=AsinhStretch()))
ax[1].imshow(map_2.data, cmap='Greens_r',norm=ImageNormalize(vmin=1,
                                             vmax=np.percentile(map_2.data, 99.5),
                                             stretch=AsinhStretch()))
ax[2].imshow(map_3.data,cmap='hot',norm=ImageNormalize(vmin=1,
                                             vmax=np.percentile(map_3.data, 99.5),
                                             stretch=AsinhStretch()))



*   Try different wavelengths to see how the images compare.



### Discussion and Exploration.
Explore the dynamics of the Sun by look at imagery over different dates a few days around May 8th. Use the GOES X-ray data [(see that notebook)](https://hss2025.heliophysics.net/hub/user-redirect/lab/tree/scratch_space/NickG/Event%20Lab/Solar%20Notebooks/GOES_XRay.ipynb) to guide what dates and times to look at.
- Which wavelengths are best for identifying flares?
- Identify the location of the solar flare on the Sun's surface.
- Save Images and dates that show flares.  

Compare the features you see on these images to the ones in the white light, magnetogram, and coronagraph notebooks. Be sure that the dates are set to the same day.  

### Exploring the Solar Cycle.
Solar Features change through out the 11 year solar cycle. The Space Weather Prediction Center ([SWPC](https://www.swpc.noaa.gov/))
at NOAA tracks the [number of sunspots](https://www.swpc.noaa.gov/products/solar-cycle-progression) to show where we are in the cycle.

May of 2024 was in the middle of a solar maximum, with a lot of solar activity.  Choose another date when the solar activity might be relatively low. Copy the code from the previous cell, adjust the date and compare the two sets of images after the cell completes.  Do the same for all of the notebooks.